# Handwritten Text Recognition (CRNN)

Run the cell below to start training.

In [ ]:
# Run training
%cd ..
%run train.py

## Test Predictions

After training, test the model:

In [ ]:
import os
import sys
import torch
import pandas as pd
from PIL import Image
from IPython.display import display

sys.path.append('..')
from src.model import CRNN
from src.utils import decode_predictions
from src.dataset import IAMDataset

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('checkpoints/best_model.pth', map_location=device)

char_to_idx = checkpoint['char_to_idx']
idx_to_char = checkpoint['idx_to_char']
model = CRNN(len(char_to_idx) + 1).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded model from epoch {checkpoint.get('epoch', 'unknown')}")
print(f"Validation word accuracy: {checkpoint.get('word_acc', 0):.2%}")
print('-' * 50)

# Test on validation samples
val_df = pd.read_csv('dataset/processed/val.csv')
dataset = IAMDataset(val_df.iloc[:10], char_to_idx, 128, 32)

for i in range(5):
    img, label = dataset[i]
    img_batch = img.unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred = model(img_batch)
        pred_text = decode_predictions(pred, idx_to_char)[0]
    
    true_text = ''.join([idx_to_char[l.item()] for l in label])
    match = '✓' if pred_text == true_text else '✗'
    print(f"{match} Predicted: '{pred_text}' | True: '{true_text}'")